<div style="background-color:#DFC7F2; padding:15px; margin:15px;">


**Центр непрерывного образования НИУ ВШЭ**

## Программа «Python для автоматизации и анализа данных»

### Занятие 10. Часть 4. Скачивание файлов

*Автор: Маргарита Бурова, НИУ ВШЭ*

</div>

## Скачивание файлов

Ещё одно применение скрейпинга, о котором мы пока не поговорили, — это автоматическое скачивание файлов. Например, вы хотите скачать научные статьи по определённому ключевому слову или прайс-листы поставщика, которые он выкладывает на сайт в Excel.

Давайте посмотрим, как скачивать файлы, на примере pdf и заодно попробуем походить по ссылкам. Кстати, этот процесс ещё часто называют spidering или crawling, потому что ваш скрипт, как паучок, ползёт по ссылкам (отсюда и название первых поисковых роботов — spider).

Давайте попробуем сохранить англоязычные summary диссертаций, защищённых в 2019 году:

https://www.hse.ru/sci/diss/?author=&chief=&year=2019&type=1&degree_type=&council=&spec=&fulltext=yes

In [ ]:
import requests
from bs4 import BeautifulSoup
from time import sleep
import pandas as pd

In [ ]:
link = 'https://www.hse.ru/sci/diss/?author=&chief=&year=2019&type=1&degree_type=&council=&spec=&fulltext=yes'

In [ ]:
soup = BeautifulSoup(requests.get(link).text)

Давайте для начала поэкспериментируем с первым кандидатом.

In [ ]:
soup

In [ ]:
print(len(soup.find_all('a')))
soup.find_all('a', {'class':'link'})[:50]

Видим, что ссылок очень много, а нам нужны только те, которые ведут на summary. Можно поискать их по тексту ссылки.

In [ ]:
for link in soup.find_all('a', string='Summary'):
    print(link.get('href'))

И соберём ссылки в список.

In [ ]:
links = []
for link in soup.find_all('a', string='Summary'):
    print(link)
    links.append(link.get('href'))

print(links)

Отлично. Теперь нужно придумать, откуда взять название для каждого файла. Пусть это будут фамилии авторов — давайте доберёмся до них. Такую задачу мы ещё не решали: будем искать тег по тексту, а потом обращаться к его родителю (потому что ни ячейку таблицы с именем автора, ни саму таблицу не получится однозначно выделить).

In [ ]:
for author in soup.find_all('div', {'class':'p1 v'}):
    try:
        print(author.span.text)
    except:
        pass

In [ ]:
soup.find_all('div', string='Соискатель:')[0].parent()[1].get_text()

Достанем фамилии.

In [ ]:
authors = []
for author in soup.find_all('div', string='Соискатель:'):
    print(author.parent()[1].get_text().split()[0])
    authors.append(author.parent()[1].get_text().split()[0])

print(authors)

Проверим, что списки действительно одинаковой длины.

In [ ]:
len(links) == len(authors)

In [ ]:
print(links)

Теперь попробуем сохранить файл. У нас все файлы pdf, будем называть их по фамилии автора. Кстати, если файлы разного формата, то тип можно узнать из заголовков ответа (`headers`):

In [ ]:
requests.get(links[0]).headers['content-type']

In [ ]:
# stream=True - читаем файл постепенно, потому что pdf может быть большим и не поместиться в память
summary = requests.get(links[0], stream=True)

# на всякий случай проверяем тип, иначе можем получить битый файл
if summary.headers['content-type'] == 'application/pdf':

    # wb - запись байтовой информации
    fh = open('test.pdf', 'wb')

    # записываем туда «содержимое» файла по ссылке
    fh.write(summary.content)
    fh.close()

In [ ]:
import os
os.getcwd()

Давайте теперь ещё добавим имя файла по фамилии.

In [ ]:
# stream=True - читаем файл постепенно, потому что pdf может быть большим и не поместиться в память
summary = requests.get(links[0], stream=True)

# на всякий случай проверяем тип, иначе можем получить битый файл
if summary.headers['content-type'] == 'application/pdf':

    # wb - запись байтовой информации
    fh = open(f'{authors[0]}.pdf', 'wb')

    # записываем туда «содержимое» файла по ссылке
    fh.write(summary.content)
    fh.close()

Давайте сохраним обработку файла в функцию и скачаем уже всё в цикле.

In [ ]:
authors

In [ ]:
def get_pdf(idx):
    summary = requests.get(links[idx], stream=True) # читаем файл постепенно, потому что pdf может быть большим и не поместиться в память
    if summary.headers['content-type'] == 'application/pdf': # на всякий случай проверяем тип, иначе можем получить битый файл
        fh = open(f'{authors[idx]}.pdf', 'wb') # wb - запись байтовой информации
        fh.write(summary.content) # записываем туда «содержимое» файла по ссылке
        fh.close()

In [ ]:
link = 'https://www.hse.ru/sci/diss/?author=&chief=&year=2019&type=1&degree_type=&council=&spec=&fulltext=yes'

soup = BeautifulSoup(requests.get(link).text)

links = []
for link in soup.find_all('a', string='Summary'):
    links.append(link.get('href'))

authors = []
for author in soup.find_all('div', string='Соискатель:'):
    authors.append(author.parent()[1].get_text().split()[0])

In [ ]:
len(links)

In [ ]:
len(authors)

In [ ]:
for idx in range(len(links)):
    get_pdf(idx)
    sleep(1) # небольшая пауза, чтобы не нагружать сайт

Готово!